### Faire la premiere methode : le TF-IDF + Logistic Regression + Linear SVM

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score, classification_report
from preprocessing import split_data, preprocessing

In [ ]:
df = preprocessing("pcm")

In [ ]:
X_train, y_train, _,_,X_test, y_test = split_data(df)

In [ ]:
X_train

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# 1. Définition de ton pipeline de base
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        max_features=10000,
        sublinear_tf=True,
        min_df=2,
        max_df=0.95
    )),
    ("logreg", LogisticRegression(
        l1_ratio=0,
        solver='lbfgs',
        max_iter=1000,
        class_weight=None
    ))
])

# 2. Définition de la grille d'hyperparamètres à tester via Validation Croisée
# Nous testons ici différentes forces de régularisation (C)
param_grid = {
    'logreg__C': [0.1, 1.0, 10.0]
}

# 3. Configuration de la Validation Croisée (ex: 5-Fold) sur le Train set
# GridSearchCV va diviser X_train en 5 blocs, et appliquer le pipeline correctement
grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    verbose=1,
    n_jobs=-1 # Utilise tous les cœurs du processeur pour aller plus vite
)

# 4. Exécution de la validation croisée et entraînement final
print("Entraînement avec validation croisée en cours...")
grid_search.fit(X_train, y_train)

# Affichage des meilleurs résultats trouvés sur le Train set
print(f"\nMeilleur score de validation croisée : {grid_search.best_score_:.4f}")
print(f"Meilleur paramètre trouvé : {grid_search.best_params_}")

# 5. Évaluation finale et unique sur le jeu de TEST
# grid_search.best_estimator_ contient le modèle automatiquement réentraîné sur TOUT X_train
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_test)

print("\n=== RAPPORT D'ÉVALUATION FINALE (SUR LE JEU DE TEST) ===")
print(classification_report(y_test, y_pred_test))